# SemEval 2025 Task 9: The Food Hazard Detection Challenge

---
> Eleni Kechrioti <br />
> Department of Informatics <br />
> Athens University of Economics and Business <br />
> p3210078@aueb.gr

## Introduction

The assignment is structured as follows to ensure clarity and ease of navigation:

- **Root Directory**:
  - **`SemEval 2025 Task 9.ipynb`**: Jupyter Notebook file where the analysis is performed.
  - **`incidents_train.csv`**: The training dataset containing incident data, including titles, descriptions, and corresponding labels for classification tasks.
  - **`incidents_test.csv`**: The test dataset used to evaluate the performance of the trained model.
  - **`incidents_valid.csv`**: The validation dataset.
  - **`/submission`**: A folder containing the total predictions on the validation and testing datasets.

*-Note:* The datasets and folder will not appear when you download the notebook in the folder, but later on, as we will download them from the [SemEval's github page](https://github.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io).

*--Note:* For this analysis I used the `text` column as I find that due to the larger description, i.e more information, it gives better results.

<br><br>
### Step 1: Install Required Libraries
For this analysis we will use 

- **scikit-learn**: A machine learning library for Python, providing simple tools for data analysis and modeling, such as classification, regression, and clustering algorithms.
- **TensorFlow**: An open-source machine learning library used to build and train models, especially deep learning architectures like BERT.
- **imblearn**: A library for imbalanced learning, which includes various tools for resampling datasets and handling imbalanced class distributions.
- **PyTorch**: A deep learning library widely used for its flexibility and ease of use in building neural network models.


In [ ]:
%pip install scikit-learn tensorflow imbalanced-learn torch

Once you have done that you can go ahead and run the notebook and start building and evaluating the machine learning models. We will use the following libraries and modules are imported to help in various stages of the machine learning pipeline, including data preprocessing, model training, and evaluation:

- **Pandas** and **NumPy**: For data manipulation and numerical operations.
- **LabelEncoder**: For encoding categorical labels into numeric values for classification tasks.
- **StandardScaler**: For scaling features to standardize the data, improving model performance.
- **RandomForestClassifier**: A powerful ensemble learning method for classification tasks.
- **TensorFlow** and **PyTorch**: Deep learning frameworks for implementing neural networks (though PyTorch is not utilized here).
- **Pipeline**: A utility from scikit-learn that allows chaining transformers and classifiers into a single object for streamlined training and evaluation.
- **ConfusionMatrixDisplay** and **classification_report**: For evaluating model performance with metrics like accuracy, precision, recall, and F1-score.
- **TfidfVectorizer**: For converting text data into numerical features using TF-IDF, which is often used in text classification tasks.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
import random
import tensorflow as tf
import torch
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

### Step 2: Download the data

Let's download our data and take a look at them.

In [2]:
# download training data (labeled):
!curl -O https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_train.csv

# download testing data (labeled):
!curl -O https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_test.csv

# download validation data (labeled):
!curl -O https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_valid.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 12.2M  100 12.2M    0     0  11.0M      0  0:00:01  0:00:01 --:--:-- 11.0M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 2479k  100 2479k    0     0  3683k      0 --:--:-- --:--:-- --:--:-- 3705k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0 1337k    0  2756    0     0   6163      0  0:

### Step 3: Understand the data

In this step, we will load the data from the provided CSV files into variables `trainset`, `testset`, and `validset` respectively. These datasets contain information about food recall incidents, which will be used for classification tasks. The structure of the dataset as you can see from the sample below includes several important columns:

- **year**: The year when the recall was issued.
- **month**: The month when the recall was issued.
- **day**: The day when the recall was issued.
- **country**: The country code where the recall was issued.
- **title**: A brief title describing the recall incident.
- **text**: A detailed description of the recall incident, containing information about the affected products, health hazards, and other relevant details.
- **hazard-category**: The category of the hazard involved in the recall (e.g., allergens, biological, fraud).
- **product-category**: The category of the product affected by the recall (e.g., meat, dairy products).
- **hazard**: A more specific description of the hazard involved (e.g., fish, E. coli).
- **product**: The name or type of the recalled product (e.g., chicken-based products, ground beef).



In [140]:
trainset = pd.read_csv('incidents_train.csv', index_col=0, encoding='utf-8')
trainset.sample(5)

,year,month,day,country,title,text,hazard-category,product-category,hazard,product
4371,2020,8,13,us,Serafin Fishery Issues Allergy Alert on Undecl...,Serafin Fishery is recalling its 8-ounce conta...,allergens,"soups, broths, sauces and condiments",fish and products thereof,dip-sauce
1514,2016,2,8,us,Living Tree Community Foods Recalls Macadamia ...,"Living Tree Community Foods of Berkeley, Calif...",biological,"nuts, nut products and seeds",salmonella,macadamia nuts
2120,2017,3,1,us,"Wayne Farms, LLC Recalls Ready-To-Eat Chicken ...","WASHINGTON, Feb. 28, 2017 – Wayne Farms, LLC, ...",other hazard,"meat, egg and dairy products",processing,chicken based products
3660,2019,8,1,us,Bimbo Bakeries USA Voluntary Recall of Entenma...,"null Bimbo Bakeries USA, Inc. has initiated a ...",foreign bodies,cereals and bakery products,plastic fragment,cookies
858,2013,7,11,us,2009 - Frito-Lay Issues Nationwide Voluntary R...,"FOR IMMEDIATE RELEASE -- PLANO, TX (March 31, ...",biological,"nuts, nut products and seeds",salmonella,pistachio nuts


In [141]:
# load test data:
testset = pd.read_csv('incidents_test.csv', index_col=0)

testset.sample(3)

,year,month,day,country,title,text,hazard-category,product-category,hazard,product
520,2018,3,29,us,Target Corporation Recalls Frozen Ready-To-Eat...,"WASHINGTON, March 29, 2018 – Target Corporatio...",other hazard,"meat, egg and dairy products",improper conditions,other not classified meat products
325,2016,4,20,uk,Argo Poultry recalls Whole Cooked Chickens due...,Argo Foods has instigated a recall as it has b...,fraud,"meat, egg and dairy products",incorrect use by dates,chicken based products
700,2019,10,12,us,"YOUBITE, LLC Recalls Pork Sausage and Turkey S...","WASHINGTON, Oct. 11, 2019 – YOUBITE, LLC, a Ca...",fraud,"meat, egg and dairy products",incorrect labeling,turkey based products


In [142]:
# load validation data:
validset = pd.read_csv('incidents_valid.csv', index_col=0)

validset.sample(3)

,year,month,day,country,title,text,hazard-category,product-category,hazard,product
55,2007,8,2,au,Woolworths Limited—Homebrand Oyster Sauce,PRA No. 2007/9432 Date published 2 Aug 2007 Pr...,allergens,"soups, broths, sauces and condiments",cereals containing gluten and products thereof,oyster sauce
260,2017,4,22,ca,Longo's brand Ontario Lean Ground Veal recalle...,Food Recall Warning - Longo's brand Ontario Le...,biological,"meat, egg and dairy products",escherichia coli,ground beef
367,2019,3,5,au,Peter Bouchier Thai Chicken Stir Fry,Page Content ​Thai Chicken Stir Fry approximat...,allergens,prepared dishes and snacks,peanuts and products thereof,cooked chicken


### Step 4: Data Preprocessing

Before we start modeling, we need to understand the data's distribution and handle missing values, if any. Additionally, we can check the class distribution for different labels, as it will help us decide on techniques for handling imbalanced datasets (e.g., oversampling, undersampling).

This step ensures that the data is clean, consistent, and ready for machine learning tasks. After understanding the data, we can move on to feature extraction and model building.

In [6]:
_ = trainset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5082 entries, 0 to 5983
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   year              5082 non-null   int64 
 1   month             5082 non-null   int64 
 2   day               5082 non-null   int64 
 3   country           5082 non-null   object
 4   title             5082 non-null   object
 5   text              5082 non-null   object
 6   hazard-category   5082 non-null   object
 7   product-category  5082 non-null   object
 8   hazard            5082 non-null   object
 9   product           5082 non-null   object
dtypes: int64(3), object(7)
memory usage: 436.7+ KB


In [7]:
_ = testset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 997 entries, 0 to 996
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   year              997 non-null    int64 
 1   month             997 non-null    int64 
 2   day               997 non-null    int64 
 3   country           997 non-null    object
 4   title             997 non-null    object
 5   text              997 non-null    object
 6   hazard-category   997 non-null    object
 7   product-category  997 non-null    object
 8   hazard            997 non-null    object
 9   product           997 non-null    object
dtypes: int64(3), object(7)
memory usage: 85.7+ KB


In [8]:
_ = validset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 565 entries, 0 to 564
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   year              565 non-null    int64 
 1   month             565 non-null    int64 
 2   day               565 non-null    int64 
 3   country           565 non-null    object
 4   title             565 non-null    object
 5   text              565 non-null    object
 6   hazard-category   565 non-null    object
 7   product-category  565 non-null    object
 8   hazard            565 non-null    object
 9   product           565 non-null    object
dtypes: int64(3), object(7)
memory usage: 48.6+ KB


We can notice from the above that there are not any values missing from our datasets. This means that we do not need to perform any imputation or removal of missing data. All the required columns, such as `title` and `text`, are present, ensuring that we have a complete dataset for training, validation, and testing. 

However as we can see below our dataset shows an imbalanced distribution of categories, with certain hazards and products significantly outnumbering others. The same goes for the columns hazard and product, if you follow the same calculations for those, but it is not shown here as the output is significantly large in rows (128 and 1022 respectively).

In [9]:
from IPython.display import display, HTML

# Get the value counts
hazard_category_counts = trainset['hazard-category'].value_counts().to_frame().rename(columns={'hazard-category': 'Count'})
product_category_counts = trainset['product-category'].value_counts().to_frame().rename(columns={'product-category': 'Count'})

# Display both dataframes side by side using HTML table formatting
display(HTML(f"<table><tr><td>{hazard_category_counts.to_html()}</td><td>{product_category_counts.to_html()}</td></tr></table>"))


,count
hazard-category,
allergens,1854
biological,1741
foreign bodies,561
fraud,371
chemical,287
other hazard,134
packaging defect,54
organoleptic aspects,53
food additives and flavourings,24


Due to this imbalance you can propably guess that any model will be trained very well for categories like `allergens` and `meat, egg and dairy products` but not to others. 

Since there is a significant imbalance in the number of samples for each hazard category, this could lead to poor model performance for the underrepresented classes. There are several ways to address this imbalance:

1. Resampling Techniques:
    - Over-sampling the minority classes (e.g., using SMOTE).
    - Under-sampling the majority classes.
    - A combination of both methods.

2. Class Weight Adjustment:
    - Assign higher weights to the minority classes during model training, so the model will penalize mistakes on minority classes more heavily. Many models (such as Random Forest and Neural Networks) allow you to specify class weights.

Here we will set the "seed", before we start, in various libraries to ensure the reproducibility of the results. When you set the same seed for the random number generator, you will get the same results every time you run the code. Why do we need it though?

- Random processes: Many machine learning algorithms, including Random Forests, neural networks, and others, involve some random processes, such as:
    - Splitting data into training and validation sets
    - Initializing weights in neural networks
    - Bootstrapping samples in Random Forest
- These processes can lead to slightly different results every time you run the code, even if everything else is identical.
- By setting a fixed seed, you ensure that these random processes are initialized in exactly the same way every time, leading to identical results across different runs. This makes it easier to compare models and results.


In [117]:
# Set the seed in all libraries
np.random.seed(30)
random.seed(30)
tf.random.set_seed(30)
torch.manual_seed(30)

Also, before we start here is the evaluation function we will use to evaluate our results, by calculating the macro-F1-score on the predicted labels (hazards_pred & products_pred) using the annotated labels (hazards_true & products_true) as ground truth. 

In [162]:
from sklearn.metrics import f1_score

# Function to compute F1 scores for hazards and products
def compute_score(hazards_true, products_true, hazards_pred, products_pred):
    # Compute F1 score for hazards:
    f1_hazards = f1_score(
        hazards_true,  # True labels for hazards
        hazards_pred,  # Predicted labels for hazards
        average='macro'  # Using macro average for F1 score
    )
    # Compute F1 score for products, considering only the cases where the hazard prediction is correct
    f1_products = f1_score(
        products_true[hazards_pred == hazards_true],  # True labels for products where the hazard prediction is correct
        products_pred[hazards_pred == hazards_true],  # Predicted labels for products where the hazard prediction is correct
        average='macro'  # Using macro average for F1 score
    )
    # Return the average of both F1 scores (for hazards and products)
    return (f1_hazards + f1_products) / 2.


## SubTask 1: Hazard and Product category classification

In SubTask 1, we want to predict the type of the hazard and the product. This is a multi-class classification with a single label. Here we will introduce another solution to handle the imbalance of the data. Why don’t we use one of the others mentioned previously?

I did test them, and with every single one I faced difficulties.

- **SMOTE** uses the `k_neighbors` parameter to find the k nearest neighbors and generate synthetic samples based on those neighbors. However, when the dataset is highly imbalanced with a majority class overwhelming others, SMOTE may lead to overfitting, as it generates new samples that may not accurately reflect the minority class distribution. The main issue though here, was that many categories had only one sample and SMOTE could not find the nearest neighbors (which is minimum 2).

- **Under-sampling** method can also be problematic. When trying to under sample the majority classes, I was led to a loss of valuable data from the majority class. Thus, the prediction accuracy was significantly decreasing.

- **Class weights** in classifiers attempt to address the imbalance by giving more importance to the minority class. While this can be effective, it does not always guarantee improved performance, especially when the imbalance is extreme. This was the case here.

Given these challenges, I decided to leverage the **OneVsRest (OvR)** strategy with **RandomForestClassifier** as the base model.

### OneVsRest Classifier for Multi-Class Classification

#### Why OneVsRest?

The OvR strategy trains a separate classifier for each class and handles class imbalance naturally by focusing each classifier on distinguishing a single class from the rest. This avoids the problems of generating synthetic data or overfitting while making the model more robust and interpretable.

- **Simplifies Multi-Class Problem**: Breaks it down into binary classification tasks, making it easier to implement and interpret.
- **Handles Class Imbalance**: Each classifier focuses on distinguishing a single class from the rest, reducing the impact of class imbalance.
- **Efficient**: Easy to implement and computationally efficient, especially with models like **RandomForest** that can handle large feature spaces and provide built-in class balancing.

It is worth mentioning here, that this is not a usual implementation as the OvR strategy is commonly used for multi-label classification and here we have single-label. Although, for all the reasons mentioned above we can use it here.

#### Overview
The code uses the **OneVsRest** strategy for multi-class classification with a **RandomForestClassifier**. Each class has its own classifier, and the final prediction is made based on which classifier is most confident about a sample.

#### Key Components

1. **TfidfVectorizer**:  
   Converts text data (titles) into numerical features using n-grams (2-5 characters). It applies **TF-IDF** weighting to highlight important words while filtering out common or rare terms.

2. **Pipeline**:  
   A `Pipeline` is used to chain the vectorization and classification steps. This ensures smooth data transformation and classification in a single process.

3. **OneVsRestClassifier**:  
   A separate binary classifier is trained for each class. The classifier predicts whether a sample belongs to its class or not, and the final prediction is the class with the highest probability.

4. **RandomForestClassifier**:  
   A **RandomForest** model is used as the base classifier for each binary task.


This code efficiently tackles multi-class classification by using **OneVsRest** with **RandomForest** as the base classifier. It’s especially useful when dealing with imbalanced datasets, as each class gets its own dedicated classifier, and class imbalance is managed through **class weights**.


In [189]:
def train_ovr_classifier(trainset, validset, label):
    # TfidfVectorizer to turn titles into numerical values
    vectorizer = TfidfVectorizer(strip_accents='unicode', analyzer='char', ngram_range=(2,5), max_df=0.5, min_df=5)
    
    # Pipeline with OneVsRestClassifier and RandomForest
    text_clf_ovr = Pipeline([
        ('vect', vectorizer),
        ('clf', OneVsRestClassifier(RandomForestClassifier(n_estimators=200, random_state=30)))
    ])
    
    # Train the model
    text_clf_ovr.fit(trainset['text'], trainset[label])
    
    # Predict
    y_pred = text_clf_ovr.predict(validset['text'])
    
    # Print the classification Report
    print(classification_report(validset[label], y_pred, zero_division=0))
    
    return y_pred  # Return the predictions

Let's see how our algorithm performs

In [ ]:
hazard_cat_preds = train_ovr_classifier(trainset, validset, 'hazard-category')
product_cat_preds = train_ovr_classifier(trainset, validset, 'product-category')


                                precision    recall  f1-score   support

                     allergens       0.93      1.00      0.96       207
                    biological       0.97      0.99      0.98       194
                      chemical       0.86      0.89      0.88        28
food additives and flavourings       1.00      0.50      0.67         2
                foreign bodies       0.88      1.00      0.93        63
                         fraud       0.93      0.61      0.74        41
          organoleptic aspects       0.75      0.38      0.50         8
                  other hazard       0.89      0.57      0.70        14
              packaging defect       0.50      0.12      0.20         8

                      accuracy                           0.93       565
                     macro avg       0.86      0.67      0.73       565
                  weighted avg       0.92      0.93      0.92       565

                                                   precision 

### Performance Summary

- **Hazard Categories**: The model performs well on larger categories like `allergens` and `biological`, with high F1-scores (0.96 and 0.98). However, it struggles with rare categories, like `packaging defect`, where F1-scores is 0.20 due to very few samples. However we can see that the algorithm worked for some smaller categories like `food additives and flavourings` that has only 2 samples compared to the 200+ of others and 0.68 on F1 score.
  
- **Product Categories**: Similar to hazard categories, performance is strong for some classes like `alcoholic beverages` (F1 = 0.83) but weak for others with limited samples, like `feed materials`, which gets an F1-score of 0.00.

- **Overall F1 Score**: The macro F1-score is 0.73 for hazard categories and 0.65 for product categories, indicating moderate performance, especially for smaller classes. The weighted F1-score is higher, reflecting the impact of larger classes.

- **General Observations**: The class imbalance significantly impacts performance, with the model performing better on classes with more samples. 

The overall F1-score for SubTask 1 on the valid dataset is 0.72, which is considered well, taking in mind the high number of classes and the imbalance.


In [174]:
print('Score Sub-Task 1:',compute_score(validset['hazard-category'], validset['product-category'], hazard_cat_preds, product_cat_preds))

Score Sub-Task 1: 0.7211688384041834


On the test dataset we see similar results, if we examine each category. The overall performance is 0.65, with hazard category being 0.68 and product category 0.6, which can be expected as the dataset is almost double the size of the validset.

In [175]:
hazard_cat_preds_test = train_ovr_classifier(trainset, testset, 'hazard-category')
product_cat_preds_test = train_ovr_classifier(trainset, testset, 'product-category')

                                precision    recall  f1-score   support

                     allergens       0.94      0.98      0.96       365
                    biological       0.96      0.98      0.97       343
                      chemical       0.94      0.94      0.94        52
food additives and flavourings       1.00      0.25      0.40         4
                foreign bodies       0.88      0.99      0.93       111
                         fraud       0.85      0.67      0.75        75
                     migration       0.00      0.00      0.00         1
          organoleptic aspects       1.00      0.40      0.57        10
                  other hazard       0.71      0.58      0.64        26
              packaging defect       1.00      0.50      0.67        10

                      accuracy                           0.93       997
                     macro avg       0.83      0.63      0.68       997
                  weighted avg       0.93      0.93      0.93 

In [176]:
print('Score Sub-Task 1:',compute_score(testset['hazard-category'], testset['product-category'], hazard_cat_preds_test, product_cat_preds_test))

Score Sub-Task 1: 0.6449519953891107


## SubTask2: Hazard and Product vector classification

In this SubTask we want to predict the exact hazard and product. We will use the same training and predicting algorithm as previously.

The results here are expected to be extremely low, but it can be expected from the reasons we said before. It is also very difficult to expect good results when handling 128 and 1022 categories.

In [ ]:
hazard_preds = train_ovr_classifier(trainset, validset, 'hazard')

                                                 precision    recall  f1-score   support

                                      Aflatoxin       1.00      1.00      1.00         1
                                 abnormal smell       0.00      0.00      0.00         1
                                      allergens       0.00      0.00      0.00         1
                                         almond       0.83      0.71      0.77         7
                         antibiotics, vet drugs       0.00      0.00      0.00         1
                                  bacillus spp.       0.67      1.00      0.80         2
                           bad smell / off odor       0.00      0.00      0.00         1
                                  bone fragment       0.00      0.00      0.00         1
                                     brazil nut       0.00      0.00      0.00         1
                              bulging packaging       0.50      0.33      0.40         3
                    

As we can see above it actually does a preety good job predicting the type of hazard. In general 0.52 in F1 score is not good, but given the extrimity of the imbalance and the large amount of categories it seems well enough. The problem here is for the product prediction as the F1 score is only 0.21.

In [180]:

product_preds = train_ovr_classifier(trainset, validset, 'product')

                                                      precision    recall  f1-score   support

                              Catfishes (freshwater)       1.00      1.00      1.00         2
                                     Dried pork meat       0.00      0.00      0.00         1
                               Fishes not identified       0.40      0.40      0.40         5
                 Precooked cooked pork meat products       0.00      0.00      0.00         1
                                     alfalfa sprouts       0.33      1.00      0.50         1
                                               algae       0.00      0.00      0.00         0
                                         almond milk       1.00      1.00      1.00         1
                                     almond products       0.00      0.00      0.00         0
                                             almonds       0.00      0.00      0.00         0
                                           appetizer       

Bringing the total score of this subtask to 0.38 

In [ ]:
print('Score Sub-Task 2:',compute_score(validset['hazard'], validset['product'], hazard_preds, product_preds))

Score Sub-Task 2: 0.3758394665645924


For the test dataset we also see simiral results as the valid dataset whith only a smaller decrease of 0.06 in the total score.

In [197]:
hazard_preds_test = train_ovr_classifier(trainset, testset, 'hazard')

                                                 precision    recall  f1-score   support

                                      Aflatoxin       1.00      1.00      1.00         2
                                 abnormal smell       0.00      0.00      0.00         1
                                alcohol content       0.00      0.00      0.00         1
                                      alkaloids       0.00      0.00      0.00         1
                                      allergens       0.00      0.00      0.00         3
                                         almond       0.82      0.69      0.75        13
                                      amygdalin       0.00      0.00      0.00         1
                         antibiotics, vet drugs       1.00      1.00      1.00         1
                                  bacillus spp.       1.00      1.00      1.00         3
                           bad smell / off odor       0.00      0.00      0.00         1
                    

In [198]:
product_preds_test = train_ovr_classifier(trainset, testset, 'product')

                                                               precision    recall  f1-score   support

                                       Catfishes (freshwater)       0.40      0.67      0.50         3
                                              Dried pork meat       0.00      0.00      0.00         1
                                        Fishes not identified       0.42      0.62      0.50         8
                                     Not classified pork meat       0.00      0.00      0.00         3
                          Precooked cooked pork meat products       0.00      0.00      0.00         1
                                            Saurida (generic)       0.00      0.00      0.00         1
                           Torpedo-shaped catfishes (generic)       0.00      0.00      0.00         0
                                                Veggie Burger       0.50      1.00      0.67         1
                                              adobo seasoning       0.00

In [199]:
print('Score Sub-Task 2:',compute_score(testset['hazard'], testset['product'], hazard_preds_test, product_preds_test))

Score Sub-Task 2: 0.3151205178410622


Finally, we can make csv files for our valid and test dataset with the real values and the predicted in a zip folder named submission.

In [ ]:
import os
from shutil import make_archive

testset['hazard-category'] = hazard_cat_preds_test
testset['product-category'] = product_cat_preds_test
testset['hazard'] = hazard_preds_test
testset['product'] = product_preds_test
validset['hazard-category'] = hazard_cat_preds
validset['product-category'] = product_cat_preds
validset['hazard'] = hazard_preds
validset['product'] = product_preds

# save predictions to a new folder:
os.makedirs('./submission/', exist_ok=True)
testset[['hazard-category', 'product-category', 'hazard', 'product']].to_csv('./submission/submission_test.csv')
validset[['hazard-category', 'product-category', 'hazard', 'product']].to_csv('./submission/submission_valid.csv')

make_archive('./submission', 'zip', './submission')

'c:\\Users\\eleni\\Desktop\\amm\\food_hazard\\submission.zip'